<a href="https://colab.research.google.com/github/IssarapongB/Data-science/blob/main/%E0%B8%9B%E0%B8%A3%E0%B8%B0%E0%B8%AA%E0%B8%9E%E0%B8%81%E0%B8%B2%E0%B8%A3%E0%B8%93%E0%B9%8C%E0%B8%A7%E0%B8%B4%E0%B8%8A%E0%B8%B2%E0%B8%8A%E0%B8%B5%E0%B8%9E1_68_5_Chat_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ก่อน run ด้วย Colab - เปลี่ยน runtime type เป็น 2025.07 นะครับ
--------------------------------------------------------
หลังจากนั้นให้ติดตั้ง library ดังนี้
1. transformers
ไลบรารีจาก Hugging Face สำหรับใช้งานโมเดล Deep Learning สำเร็จรูป
เช่น
- Text Generation
- Text Classification
- Question Answering
- Embedding / Sentence Transformer

2. accelerate
ไลบรารีที่ช่วยจัดการการรันโมเดลให้มีประสิทธิภาพมากขึ้น
รองรับการใช้งานบน CPU, GPU และการปรับแต่งทรัพยากรอัตโนมัติ

In [3]:
!pip -q install transformers accelerate

ตัวอย่าง Chatbot ด้วย Transformers
-----------------------------------------------------

    chatbot = pipeline(
       "text2text-generation",
        model="google/flan-t5-base"
    )

สร้างโมเดล Chatbot โดยใช้โมเดล FLAN-T5 ซึ่งสามารถรับข้อความและตอบกลับเป็นข้อความได้

    while True:
        user = input("You: ")


วนลูปรับข้อความจากผู้ใช้แบบต่อเนื่อง

    if user.lower() in ["exit", "quit"]:
        break


พิมพ์ exit หรือ quit เพื่อออกจากโปรแกรม

    response = chatbot(
        user,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )


ให้โมเดลสร้างคำตอบ

max_new_tokens จำกัดความยาวคำตอบ

temperature ควบคุมความสุ่มของคำตอบ (ค่ายิ่งสูง ยิ่งตอบหลากหลาย)

    print("Bot:", response[0]["generated_text"])


แสดงคำตอบที่โมเดลสร้างขึ้น

In [7]:
from transformers import pipeline

chatbot = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

while True:
    user = input("You: ")
    if user.lower() in ["exit", "quit"]:
        break

    response = chatbot(
        user,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

    print("Bot:", response[0]["generated_text"])

Device set to use cuda:0


You: exit


ตัวอย่างนี้เปลี่ยนไปใช้โมเดล Qwen2.5-1.5B-Instruct ซึ่งเป็น Large Language Model (LLM) → รองรับการสนทนาและคำสั่งที่ซับซ้อนได้ดีกว่า
---------------------------------------------------------------
ใช้ text-generation แทน text2text-generation
→ โมเดลจะ สร้างข้อความต่อเนื่องเองทั้งหมด จาก prompt

In [5]:
gen = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

def ask(user):
    prompt = f"### Instruction:\n{user}\n\n### Response:\n"
    out = gen(prompt, max_new_tokens=200, do_sample=True, temperature=0.7)
    return out[0]["generated_text"].split("### Response:\n",1)[-1].strip()

while True:
    u = input("You: ")
    if u.lower() in ["exit","quit"]:
        break
    print("Bot:", ask(u))


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


You: แมลงวัน
Bot: แมลงวันเป็นสัตว์เลี้ยงลูกด้วยนมที่มีการถ่ายทอดสายพันธุ์และสืบเชื้อมาหลายร้อยปี แมลงวันสามารถพบได้ในสภาพแวดล้อมหลากหลาย เช่น บ้านเรือน, นา, และแหล่งน้ำ พวกเขาสามารถกินอาหารหลากหลาย เช่น ผัก, ผลไม้, หอย, และแมลงอื่นๆ แมลงวันสามารถใช้เส้นทางเดิน (trail) เป็นช่องทางในการขยายพันธุ์ การเติบโตและการเจริญเติบโตของพวกเขามีความสำคัญต่อระบบนิเวศในแต่ละพื้นที่ แมลงวันเป็นสัตว์ที่มีบทบาทสำคัญในระบบน
You: quit


เพิ่ม System Prompt และ Multi-Turns
-----------------------------------------
เพิ่ม System Prompt
→ กำหนดบทบาทของบอท (อาจารย์สายวิทยาศาสตร์), รูปแบบคำตอบ, และกติกาการตอบ

รองรับ บริบทการสนทนา (Conversation History)
→ โมเดลสามารถจำบทสนทนาได้ โดยเก็บเฉพาะ 6 เทิร์นล่าสุดเพื่อลดความยาว prompt

แยกขั้นตอนสร้าง prompt (build_prompt) ออกจากการถาม (ask)
→ โค้ดอ่านง่าย และปรับแต่งโครงสร้างบทสนทนาได้สะดวก

ใช้ do_sample=False
→ ลดความสุ่ม ทำให้คำตอบนิ่งและไม่ “แต่งต่อเอง”

ใช้ return_full_text=False
→ ได้เฉพาะข้อความที่โมเดลสร้าง ไม่รวม prompt เดิม

In [6]:
from transformers import pipeline
import torch

gen = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

SYSTEM = """You are a helpful Science Prof. chatbot.
Rules:
- Reply ONLY as the Assistant.
- Do NOT write 'User:' or continue the conversation by yourself.
- Keep the answer short (1-3 sentences) unless the user asks to expand.
- If the user's question is unclear, ask ONE clarifying question.
"""

def build_prompt(user, history):
    hist = "\n".join(history[-6:])
    return f"""{SYSTEM}

Conversation:
{hist}
User: {user}
Assistant:"""

def ask(user, history):
    prompt = build_prompt(user, history)
    out = gen(
        prompt,
        max_new_tokens=80,
        do_sample=False,      # ✅ สำคัญ: กันโมเดลแต่งต่อ
        return_full_text=False
    )
    return out[0]["generated_text"].strip()

history = []

while True:
    u = input("You: ")
    if u.lower() in ["exit", "quit"]:
        break

    history.append(f"User: {u}")
    ans = ask(u, history)
    history.append(f"Assistant: {ans}")

    print("Bot:", ans)

Device set to use cuda:0


You: exit


เพิ่มขั้นตอน clean output
-----------------------------------------
เหมือนโปรแกรมที่แล้ว: มี System Prompt, รองรับ multi-turn ด้วย history 6 เทิร์น, แยก build_prompt() / ask(), ใช้ do_sample=False, และ return_full_text=False

เพิ่มขั้นตอน clean output
→ หลังโมเดลตอบแล้ว โปรแกรมจะ “ตัด” ข้อความส่วนเกินออก ถ้าโมเดลเผลอพิมพ์ต่อเอง เช่น User: หรือ Assistant:
→ ทำให้ผลลัพธ์ที่แสดงออกมาเป็น “คำตอบของบอท” ล้วน ๆ และนิ่งกว่า

In [ ]:
from transformers import pipeline
import torch

# โหลด LLM
gen = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

# System prompt
SYSTEM = """You are a helpful Science Prof. chatbot.
Rules:
- Reply ONLY as the Assistant.
- Do NOT write 'User:' or continue the conversation by yourself.
- Keep the answer short (1-3 sentences) unless the user asks to expand.
- If the user's question is unclear, ask ONE clarifying question.
"""

# สร้าง prompt
def build_prompt(user, history):
    hist = "\n".join(history[-6:])   # เก็บแค่ 6 เทิร์นล่าสุด
    return f"""{SYSTEM}

Conversation:
{hist}
User: {user}
Assistant:"""

# เรียก LLM และทำความสะอาดคำตอบ
def ask(user, history):
    prompt = build_prompt(user, history)

    out = gen(
        prompt,
        max_new_tokens=80,
        do_sample=False,             # กันโมเดลแต่งต่อ
        return_full_text=False       # เอาเฉพาะส่วนที่ generate
    )

    text = out[0]["generated_text"]

    # ---- CLEAN OUTPUT ----
    # ตัดกรณีโมเดลเผลอสร้างแท็กต่อเอง
    for stop in ["User:", "Assistant:"]:
        if stop in text:
            text = text.split(stop)[0]

    return text.strip()

# ------------------------
# Main chat loop
# ------------------------
history = []

while True:
    u = input("You: ")
    if u.lower() in ["exit", "quit"]:
        break

    history.append(f"User: {u}")
    ans = ask(u, history)
    history.append(f"Assistant: {ans}")

    print("Bot:", ans)


ติดตั้งไลบรารีสำหรับให้โมเดลเรียนรู้จากข้อความของเรา
----------------------------------------------

    pip install -U sentence-transformers faiss-cpu


คำสั่งนี้ใช้ติดตั้งไลบรารีที่จำเป็นสำหรับการทำให้โมเดล
เข้าใจและค้นหาข้อมูลจากข้อความที่เรากำหนดเอง

    sentence-transformers
ใช้แปลงข้อความเป็นเวกเตอร์ (embeddings) เพื่อวัดความหมายและความคล้ายของข้อความ

    faiss-cpu
ใช้เก็บและค้นหาเวกเตอร์อย่างรวดเร็ว
เหมาะสำหรับการสร้างระบบค้นหาข้อมูลจากเอกสารของเราเอง

📌 ขั้นตอนถัดไป เราจะใช้ไลบรารีเหล่านี้เพื่อ
ทำให้โมเดลตอบคำถามโดยอ้างอิงจากข้อความ (text) ที่เราเตรียมไว้เอง

In [8]:
pip install -U sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 45.7 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 4.1.0
    Uninstalling sentence-transformers-4.1.0:
      Successfully uninstalled sentence-transformers-4.1.0


ให้ model เรียนรู้จากฐานความรู้ของเราเอง
-----------------------------------------------

1) เพิ่ม “ฐานความรู้ของเราเอง” (docs)

เดิม: บอทตอบจากความรู้ในโมเดลล้วน ๆ

ใหม่: เราใส่ข้อความองค์กรไว้ใน docs เพื่อให้บอท “อ้างอิงจาก text ของเรา”

2) เพิ่มขั้นตอน “ทำ Embedding” ด้วย SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_vecs = embedder.encode(docs, normalize_embeddings=True)


เดิม: ไม่มีการแปลงข้อความเป็นเวกเตอร์

ใหม่: แปลงเอกสารของเราเป็น เวกเตอร์ความหมาย (embeddings) เพื่อใช้ค้นหาข้อความที่เกี่ยวข้องกับคำถาม

3) เพิ่ม “Vector Search” ด้วย FAISS
index = faiss.IndexFlatIP(dim)
index.add(doc_vecs)


เดิม: ไม่มีระบบค้นหาเอกสาร

ใหม่: ใช้ FAISS เก็บเวกเตอร์ และค้นหาเอกสารที่ใกล้กับคำถามได้เร็วมาก

IndexFlatIP = ค้นหาด้วยคะแนนแบบ inner product (ใช้ได้ดีเมื่อ normalize แล้ว)

4) มีฟังก์ชัน retrieve() เพื่อดึง “บริบท” ก่อนถาม LLM
def retrieve(query, k=3):
    ...
    return [docs[i] for i in ids[0]]


เดิม: ส่งคำถามเข้าตรง ๆ ให้ LLM

ใหม่: ค้นหาเอกสารที่เกี่ยวข้องก่อน แล้วค่อยส่งให้ LLM เป็น “context”

5) Prompt เปลี่ยนเป็น “RAG Prompt” (เพิ่ม Context)
Context:
- ...
Conversation so far:
...
User: ...
Assistant:


เดิม: มีแค่ SYSTEM + history + user

ใหม่: เพิ่มส่วน Context (เอกสารที่ดึงมา) เข้าไปใน prompt เพื่อบังคับให้ตอบจากข้อมูลของเรา

6) System Prompt เข้มขึ้น: “ใช้เฉพาะ Context เท่านั้น”
- Use ONLY the provided context to answer.
- If the answer is not in the context, say you do not know.
- Answer in Thai.


เดิม: คุมสไตล์การตอบเฉย ๆ

ใหม่: คุม “แหล่งข้อมูล” ด้วย เพื่อกันโมเดลตอบมั่ว (hallucination)

7) โครงสร้าง ask() เปลี่ยน: “retrieve → build_prompt → generate”
context = retrieve(user, k=3)
prompt = build_prompt(user, history, context)
out = gen(...)


เดิม: build_prompt แล้ว generate เลย

ใหม่: มีขั้น ดึง context ก่อนทุกครั้ง แล้วค่อยให้ LLM ตอบ

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline
import torch

# -----------------------------
# 1) ข้อมูลองค์กร (ตัวอย่าง)
# -----------------------------
docs = [

"""
สาขาวิชาวิทยาการข้อมูลของคณะวิทยาศาสตร์และเทคโนโลยี มีพันธกิจในการผลิตบัณฑิตที่มีความรู้และทักษะด้านการวิเคราะห์ข้อมูล การเขียนโปรแกรม และการประยุกต์ใช้เทคโนโลยีปัญญาประดิษฐ์ในงานจริง โดยหลักสูตรได้ออกแบบรายวิชาต่าง ๆ ให้สอดคล้องกับความต้องการของตลาดแรงงานในปัจจุบัน หนึ่งในรายวิชาหลักที่เปิดสอนคือรายวิชา “Python สำหรับวิทยาการข้อมูล” ซึ่งมุ่งเน้นการใช้ภาษา Python ในการจัดการข้อมูล การวิเคราะห์ข้อมูลเชิงสถิติ และการสร้างโมเดลเบื้องต้นทางการเรียนรู้ของเครื่อง

รายวิชาดังกล่าวครอบคลุมหัวข้อสำคัญ เช่น โครงสร้างข้อมูลพื้นฐาน การใช้ไลบรารี Pandas และ NumPy การแสดงผลข้อมูลด้วย Matplotlib และ Seaborn รวมถึงการฝึกปฏิบัติผ่านกรณีศึกษาจากข้อมูลจริง นักศึกษาที่ผ่านรายวิชานี้จะสามารถนำความรู้ไปต่อยอดในการทำโครงงานวิทยาการข้อมูล และการเรียนในรายวิชาขั้นสูงต่อไปได้
""",

"""
การขอใช้ห้องปฏิบัติการคอมพิวเตอร์ของคณะวิทยาศาสตร์และเทคโนโลยี ต้องดำเนินการตามขั้นตอนที่กำหนดไว้เพื่อให้การใช้ทรัพยากรเป็นไปอย่างมีประสิทธิภาพและเป็นธรรม ผู้ขอใช้ห้องปฏิบัติการจะต้องกรอกแบบฟอร์มขอใช้ห้องปฏิบัติการล่วงหน้า โดยระบุวัตถุประสงค์ในการใช้งาน รายวิชาที่เกี่ยวข้อง วันที่และเวลาที่ต้องการใช้งาน รวมถึงจำนวนผู้เข้าใช้งาน

หลังจากกรอกแบบฟอร์มเรียบร้อยแล้ว ผู้ขอใช้จะต้องส่งแบบฟอร์มดังกล่าวเพื่อขออนุมัติจากหัวหน้าสาขาวิชาหรือผู้รับผิดชอบห้องปฏิบัติการ เมื่อได้รับการอนุมัติแล้ว จึงจะสามารถเข้าใช้งานห้องปฏิบัติการได้ตามวันและเวลาที่กำหนด ทั้งนี้ ผู้ใช้งานต้องปฏิบัติตามระเบียบของห้องปฏิบัติการอย่างเคร่งครัด เช่น ห้ามติดตั้งซอฟต์แวร์เพิ่มเติมโดยไม่ได้รับอนุญาต และต้องดูแลอุปกรณ์ให้อยู่ในสภาพเรียบร้อย
""",

"""
นโยบายการลาของบุคลากรในคณะวิทยาศาสตร์และเทคโนโลยี ถูกกำหนดขึ้นเพื่อให้การบริหารงานบุคคลเป็นไปอย่างเหมาะสมและโปร่งใส การลาป่วยเป็นสิทธิของบุคลากรเมื่อมีเหตุจำเป็นด้านสุขภาพ โดยผู้ลาป่วยจะต้องแจ้งหัวหน้างานทราบโดยเร็วที่สุด หากการลาป่วยมีระยะเวลาไม่เกินสองวันทำการ สามารถแจ้งลาโดยไม่ต้องแนบเอกสารเพิ่มเติม

อย่างไรก็ตาม ในกรณีที่การลาป่วยมีระยะเวลาเกินสองวันทำการ ผู้ลาจะต้องแนบใบรับรองแพทย์จากสถานพยาบาลที่เชื่อถือได้ เพื่อประกอบการพิจารณา การไม่ปฏิบัติตามนโยบายการลาที่กำหนดไว้ อาจส่งผลต่อการบันทึกเวลาการปฏิบัติงานและการประเมินผลการปฏิบัติงานประจำปี
"""
]


# -----------------------------
# 2) Embedding + FAISS
# -----------------------------
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vecs = embedder.encode(docs, normalize_embeddings=True)

dim = doc_vecs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_vecs.astype("float32"))

def retrieve(query, k=3):
    qv = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, k)
    return [docs[i] for i in ids[0]]

# -----------------------------
# 3) โหลด LLM
# -----------------------------
gen = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

SYSTEM = """You are a helpful assistant for answering questions about the organization.
Rules:
- Use ONLY the provided context to answer.
- If the answer is not in the context, say you do not know.
- Answer in Thai.
- Keep the answer concise.
"""

# -----------------------------
# 4) สร้าง prompt แบบ RAG + history
# -----------------------------
def build_prompt(user, history, context):
    hist = "\n".join(history[-6:])
    ctx  = "\n- ".join(context)

    return f"""{SYSTEM}

Context:
- {ctx}

Conversation so far:
{hist}

User: {user}
Assistant:"""

def ask(user, history):
    context = retrieve(user, k=3)
    prompt = build_prompt(user, history, context)

    out = gen(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        return_full_text=False
    )

    text = out[0]["generated_text"]

    # ทำความสะอาด (กันโมเดลแถ)
    for stop in ["User:", "Assistant:"]:
        if stop in text:
            text = text.split(stop)[0]

    return text.strip()

# -----------------------------
# 5) Chat loop
# -----------------------------
history = []

while True:
    u = input("You: ")
    if u.lower() in ["exit", "quit"]:
        break

    history.append(f"User: {u}")
    ans = ask(u, history)
    history.append(f"Assistant: {ans}")

    print("Bot:", ans)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Device set to use cuda:0


ปรับเป็น Hybrid RAG
--------------------------------------------------------

1) มี “คะแนนความเกี่ยวข้อง” (similarity score) แล้วค่อยตัดสินใจ

เดิม:

retrieve() คืนแค่เอกสารที่ใกล้สุด แล้วเอาไปตอบเลยทุกครั้ง

ใหม่:

retrieve_with_scores() คืนทั้ง scores + contexts

scores, ids = index.search(qv, k)
top_score = float(scores[0])


ทำให้เรารู้ว่า “คำถามนี้เกี่ยวกับเอกสารเราจริงไหม”

2) เพิ่ม “โหมดการตอบ 2 แบบ”: RAG vs GENERAL

เดิม:

มี SYSTEM เดียว และตอบแบบ RAG เสมอ (ต้องใช้ context เท่านั้น)

ใหม่:

มี SYSTEM แยก 2 ชุด

RAG_SYSTEM สำหรับตอบเรื่ององค์กร (ต้องอ้าง context)

GENERAL_SYSTEM สำหรับตอบทั่วไป (เช่นติวภาษาไทย ฯลฯ)

และมี prompt builder แยกกัน:

build_rag_prompt(...)

build_general_prompt(...)

3) มี Threshold เพื่อสลับโหมดอัตโนมัติ

เดิม:

ไม่สนใจคะแนน → ใช้ RAG ตลอด

ใหม่:

ตั้งค่า SIM_THRESHOLD

if top_score >= SIM_THRESHOLD:
    mode = "RAG"
else:
    mode = "GENERAL"


ผลลัพธ์: ถ้าคำถามไม่เกี่ยวกับเอกสารองค์กร ระบบจะไม่ฝืนตอบจาก context (ลดตอบมั่ว/ตอบไม่ตรง)

4) ปรับความยาวคำตอบตามโหมด

เดิม:

max_new_tokens คงที่

ใหม่:

ถ้าเป็น RAG ให้ยาวขึ้นหน่อย

max_new_tokens=150 if mode == "RAG" else 100

5) ทำความสะอาดคำตอบเป็นฟังก์ชันกลาง (clean_output)

เดิม:

clean output อยู่ใน ask() โดยตรง

ใหม่:

แยกเป็นฟังก์ชัน clean_output(text) เพื่อใช้ซ้ำและอ่านง่ายขึ้น

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline
import torch

# -----------------------------
# 1) ข้อมูลองค์กร (ตัวอย่าง)
# -----------------------------
docs = [

"""
สาขาวิชาวิทยาการข้อมูลของคณะวิทยาศาสตร์และเทคโนโลยี มีพันธกิจในการผลิตบัณฑิตที่มีความรู้และทักษะด้านการวิเคราะห์ข้อมูล การเขียนโปรแกรม และการประยุกต์ใช้เทคโนโลยีปัญญาประดิษฐ์ในงานจริง โดยหลักสูตรได้ออกแบบรายวิชาต่าง ๆ ให้สอดคล้องกับความต้องการของตลาดแรงงานในปัจจุบัน หนึ่งในรายวิชาหลักที่เปิดสอนคือรายวิชา “Python สำหรับวิทยาการข้อมูล” ซึ่งมุ่งเน้นการใช้ภาษา Python ในการจัดการข้อมูล การวิเคราะห์ข้อมูลเชิงสถิติ และการสร้างโมเดลเบื้องต้นทางการเรียนรู้ของเครื่อง

รายวิชาดังกล่าวครอบคลุมหัวข้อสำคัญ เช่น โครงสร้างข้อมูลพื้นฐาน การใช้ไลบรารี Pandas และ NumPy การแสดงผลข้อมูลด้วย Matplotlib และ Seaborn รวมถึงการฝึกปฏิบัติผ่านกรณีศึกษาจากข้อมูลจริง นักศึกษาที่ผ่านรายวิชานี้จะสามารถนำความรู้ไปต่อยอดในการทำโครงงานวิทยาการข้อมูล และการเรียนในรายวิชาขั้นสูงต่อไปได้
""",

"""
การขอใช้ห้องปฏิบัติการคอมพิวเตอร์ของคณะวิทยาศาสตร์และเทคโนโลยี ต้องดำเนินการตามขั้นตอนที่กำหนดไว้เพื่อให้การใช้ทรัพยากรเป็นไปอย่างมีประสิทธิภาพและเป็นธรรม ผู้ขอใช้ห้องปฏิบัติการจะต้องกรอกแบบฟอร์มขอใช้ห้องปฏิบัติการล่วงหน้า โดยระบุวัตถุประสงค์ในการใช้งาน รายวิชาที่เกี่ยวข้อง วันที่และเวลาที่ต้องการใช้งาน รวมถึงจำนวนผู้เข้าใช้งาน

หลังจากกรอกแบบฟอร์มเรียบร้อยแล้ว ผู้ขอใช้จะต้องส่งแบบฟอร์มดังกล่าวเพื่อขออนุมัติจากหัวหน้าสาขาวิชาหรือผู้รับผิดชอบห้องปฏิบัติการ เมื่อได้รับการอนุมัติแล้ว จึงจะสามารถเข้าใช้งานห้องปฏิบัติการได้ตามวันและเวลาที่กำหนด ทั้งนี้ ผู้ใช้งานต้องปฏิบัติตามระเบียบของห้องปฏิบัติการอย่างเคร่งครัด เช่น ห้ามติดตั้งซอฟต์แวร์เพิ่มเติมโดยไม่ได้รับอนุญาต และต้องดูแลอุปกรณ์ให้อยู่ในสภาพเรียบร้อย
""",

"""
นโยบายการลาของบุคลากรในคณะวิทยาศาสตร์และเทคโนโลยี ถูกกำหนดขึ้นเพื่อให้การบริหารงานบุคคลเป็นไปอย่างเหมาะสมและโปร่งใส การลาป่วยเป็นสิทธิของบุคลากรเมื่อมีเหตุจำเป็นด้านสุขภาพ โดยผู้ลาป่วยจะต้องแจ้งหัวหน้างานทราบโดยเร็วที่สุด หากการลาป่วยมีระยะเวลาไม่เกินสองวันทำการ สามารถแจ้งลาโดยไม่ต้องแนบเอกสารเพิ่มเติม

อย่างไรก็ตาม ในกรณีที่การลาป่วยมีระยะเวลาเกินสองวันทำการ ผู้ลาจะต้องแนบใบรับรองแพทย์จากสถานพยาบาลที่เชื่อถือได้ เพื่อประกอบการพิจารณา การไม่ปฏิบัติตามนโยบายการลาที่กำหนดไว้ อาจส่งผลต่อการบันทึกเวลาการปฏิบัติงานและการประเมินผลการปฏิบัติงานประจำปี
"""
]

# -----------------------------
# 2) Embedding + FAISS
# -----------------------------
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vecs = embedder.encode(docs, normalize_embeddings=True)

dim = doc_vecs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_vecs.astype("float32"))

def retrieve_with_scores(query, k=3):
    qv = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, k)     # scores ~ cosine similarity (0..1 โดยประมาณ)
    contexts = [docs[i] for i in ids[0]]
    return scores[0], contexts

# -----------------------------
# 3) โหลด LLM
# -----------------------------
gen = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

RAG_SYSTEM = """You are a helpful assistant for answering questions about the organization.
Rules:
- Use ONLY the provided context to answer questions about the organization.
- If the answer is not in the context, say you do not know.
- Answer in Thai.
- Keep the answer concise.
"""

GENERAL_SYSTEM = """You are a helpful Thai tutor chatbot.
Rules:
- Reply ONLY as the Assistant.
- Answer in Thai.
- Keep the answer short (1-3 sentences) unless the user asks to expand.
- If the user's question is unclear, ask ONE clarifying question.
"""

# -----------------------------
# 4) สร้าง prompt แบบ Hybrid
# -----------------------------
def build_rag_prompt(user, history, context):
    hist = "\n".join(history[-6:])
    ctx  = "\n- ".join(context)
    return f"""{RAG_SYSTEM}

Context:
- {ctx}

Conversation so far:
{hist}

User: {user}
Assistant:"""

def build_general_prompt(user, history):
    hist = "\n".join(history[-6:])
    return f"""{GENERAL_SYSTEM}

Conversation so far:
{hist}

User: {user}
Assistant:"""

def clean_output(text):
    for stop in ["User:", "Assistant:"]:
        if stop in text:
            text = text.split(stop)[0]
    return text.strip()

# -----------------------------
# 5) Hybrid ask()
# -----------------------------
SIM_THRESHOLD = 0.35   # ปรับได้ (0.25-0.45 แล้วแต่เอกสาร/ภาษา)

def ask(user, history, k=3):
    scores, context = retrieve_with_scores(user, k=k)
    top_score = float(scores[0])

    # เลือกโหมดตามความเกี่ยวข้อง
    if top_score >= SIM_THRESHOLD:
        prompt = build_rag_prompt(user, history, context)
        mode = "RAG"
    else:
        prompt = build_general_prompt(user, history)
        mode = "GENERAL"

    out = gen(
        prompt,
        max_new_tokens=150 if mode == "RAG" else 100,
        do_sample=False,
        return_full_text=False
    )

    answer = clean_output(out[0]["generated_text"])

    # (optional) แสดงโหมดและคะแนน
    # print(f"[mode={mode} score={top_score:.3f}]")

    return answer, mode, top_score

# -----------------------------
# 6) Chat loop
# -----------------------------
history = []

while True:
    u = input("You: ")
    if u.lower() in ["exit", "quit"]:
        break

    history.append(f"User: {u}")
    ans, mode, score = ask(u, history, k=3)
    history.append(f"Assistant: {ans}")

    print(f"Bot: {ans}")